<a href="https://colab.research.google.com/github/Vandanareddy2/cifar10-cnn/blob/main/notebooks/cifar10-cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print(torch.cuda.get_device_name(0))

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

In [ ]:
# transform = transforms.ToTensor()
# transform = transforms.Compose([
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(32, padding=4),
#     transforms.ToTensor(),
#     transforms.Normalize((0.5, 0.5, 0.5),
#                          (0.5, 0.5, 0.5))
# ])
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

In [ ]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [ ]:
!ls data

In [ ]:
print(len(train_dataset))

In [ ]:
image, label = train_dataset[0]
print(image, label)

In [ ]:
print(image.shape)

In [ ]:
print(image[0].shape)  # Red channel
print(image[1].shape)  # Green channel
print(image[2].shape)  # Blue channel

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
image_np = image.permute(1, 2, 0)
plt.imshow(image_np)
plt.title(f"Label: {label}")
plt.show()

In [ ]:
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]
print(classes[label])


In [ ]:
import torch.nn as nn

conv1 = nn.Conv2d(
    in_channels=3,
    out_channels=6,
    kernel_size=5
)
print(conv1)

In [ ]:
image_batch = image.unsqueeze(0)
print(image_batch.shape)
output = conv1(image_batch)
print(output.shape)

In [ ]:
feature_maps = output.squeeze(0)
print(feature_maps.shape)

In [ ]:
first_map = feature_maps[0]
print(first_map.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(first_map.detach().numpy(), cmap='gray')
plt.title("Feature Map 1")
plt.colorbar()
plt.show()

In [ ]:
output = conv1(image_batch)
print(output.shape)
feature_maps = output.squeeze(0)
print(feature_maps.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(15, 3))

for i in range(6):
    axes[i].imshow(feature_maps[i].detach().numpy(), cmap='gray')
    axes[i].set_title(f"Map {i+1}")
    axes[i].axis("off")

plt.show()

In [ ]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)

pooled_output = pool(output)
print(pooled_output.shape)

In [ ]:
import torch.nn as nn

conv = nn.Conv2d(3, 6, kernel_size=5)
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size=2, stride=2)

In [ ]:
x = image.unsqueeze(0)

x = conv(x)
print("After Conv:", x.shape)

x = relu(x)
print("After ReLU:", x.shape)

x = pool(x)
print("After Pool:", x.shape)

In [ ]:
print("Model device:", next(model.parameters()).device)
print("Images device:", images.device)
print("Labels device:", labels.device)

In [ ]:
model = model.cuda()

In [ ]:
print("Model device:", next(model.parameters()).device)
print("Images device:", images.device)
print("Labels device:", labels.device)

In [ ]:
print("BEFORE TRAIN LOOP")
print(next(model.parameters()).device)

In [ ]:
# import torch.nn as nn

# class SimpleCNN(nn.Module):
#     def __init__(self):
#         super(SimpleCNN, self).__init__()

#         self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
#         self.relu = nn.ReLU()
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

#         self.flatten = nn.Flatten()
#         self.fc = nn.Linear(6 * 14 * 14, 10)

#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.relu(x)
#         x = self.pool(x)

#         x = self.flatten(x)
#         x = self.fc(x)

#         return x

import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Block 1: 32x32 → 16x16
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2: 16x16 → 8x8
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3: 8x8 → 4x4
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()

        # FINAL FEATURE SIZE = 128 * 4 * 4
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forward(self, x):

        # Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)

        # Block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool(x)

        # Block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.pool(x)

        # Classifier
        x = self.flatten(x)
        x = self.fc(x)

        return x

In [ ]:
model = SimpleCNN()

image, label = train_dataset[0]
image = image.unsqueeze(0)

output = model(image)

print(output.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

model = SimpleCNN()
model=model.cuda()

criterion = nn.CrossEntropyLoss()
print(criterion)
# optimizer = optim.SGD(model.parameters(), lr=0.01)
# optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = torch.optim.Adam(model.parameters(),lr=0.0005,weight_decay=1e-4)
print(optimizer)

In [ ]:
image, label = train_dataset[0]
image = image.unsqueeze(0).cuda()
label = torch.tensor([label]).cuda()

output = model(image)
loss = criterion(output, label)

print("Loss:", loss.item())

In [ ]:
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [ ]:


optimizer.zero_grad()

output = model(image)
loss = criterion(output, label)

loss.backward()

print(model.conv1.weight.grad.shape)

In [ ]:
print(model.conv1.weight[0].shape)


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

In [ ]:
images = images.cuda()
labels=labels.cuda()
outputs = model(images)

print(outputs.shape)

In [ ]:
print(outputs[0])

In [ ]:
predicted_class = outputs[0].argmax()

print(predicted_class)

In [ ]:
print(outputs[0])
print(outputs[0].argmax())
print(labels[0])

In [ ]:
loss = criterion(outputs, labels)

print(loss)

In [ ]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

In [ ]:
for epoch in range(30):

    running_loss = 0.0

    for images, labels in train_loader:
        images = images.cuda()
        labels = labels.cuda()

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

#Testing Phase

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.ToTensor()

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
model.eval()

In [ ]:
with torch.no_grad():
  correct = 0
  total = 0

  with torch.no_grad():
      for images, labels in test_loader:

          images = images.cuda()
          labels = labels.cuda()

          outputs = model(images)

          _, predicted = torch.max(outputs, 1)

          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  print("Test Accuracy:", 100 * correct / total)

In [ ]:
print(next(model.parameters()).device)